# Task 3: Naive Bayes
Naive Bayes learns how often each word is in ham and in spam.
Then, for a new message, it asks: is this group of words more like ham or like spam?
We try it on our three text versions and choose the best one.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
train = pd.read_csv('../data/train_prep_v2.csv')
test = pd.read_csv('../data/test_prep_v2.csv')
y_train = (train['label'] == 'spam').astype(int)
y_test = (test['label'] == 'spam').astype(int)
print('Train spam:', y_train.sum(), '| Test spam:', y_test.sum(), 'of', len(y_test))
print('If we always say ham, accuracy is', round(100 * (y_test == 0).mean(), 1), '%')

Train spam: 501 | Test spam: 125 of 1026
If we always say ham, accuracy is 87.8 %


## 2. Choosing the settings in a fair way
We must not choose the settings by looking at the test result.
If we do, we are cheating: we pick what works on the test,
and then the test is not a fair exam any more.

So we use cross-validation on the train data:
we cut the train data into 5 parts, train on 4 and test on 1,
and we do this 5 times. We take the average.
The test data is used one time only, at the end.

We also try the "alpha" setting of Naive Bayes.
Alpha is a small number the model adds to every word count,
so that a word it never saw does not get a probability of zero.
A big alpha makes the model careful; a small alpha makes it bold.

In [2]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for col in ['clean_text', 'stop_text', 'stem_text']:
    for a in [1.0, 0.5, 0.1, 0.05, 0.01]:
        pipe = make_pipeline(TfidfVectorizer(token_pattern=r'\S+'),
                             MultinomialNB(alpha=a))
        score = cross_val_score(pipe, train[col], y_train, cv=cv, scoring='f1')
        rows.append([col, a, round(100 * score.mean(), 1)])
result = pd.DataFrame(rows, columns=['text', 'alpha', 'cv_f1'])
print(result.to_string(index=False))
print('Best:')
print(result.sort_values('cv_f1', ascending=False).head(3).to_string(index=False))

      text  alpha  cv_f1
clean_text   1.00   79.3
clean_text   0.50   87.9
clean_text   0.10   92.9
clean_text   0.05   93.5
clean_text   0.01   92.7
 stop_text   1.00   84.4
 stop_text   0.50   89.8
 stop_text   0.10   93.3
 stop_text   0.05   93.4
 stop_text   0.01   92.4
 stem_text   1.00   84.8
 stem_text   0.50   90.2
 stem_text   0.10   92.9
 stem_text   0.05   93.1
 stem_text   0.01   92.6
Best:
      text  alpha  cv_f1
clean_text   0.05   93.5
 stop_text   0.05   93.4
 stop_text   0.10   93.3


## 3. Train the final model and test it
Now we use the best settings from the cross-validation.
We train on all the train data, and we test one time on the test data.

In [3]:
best_col = 'clean_text'
best_alpha = 0.05
vec = TfidfVectorizer(token_pattern=r'\S+')
X_train = vec.fit_transform(train[best_col])
X_test = vec.transform(test[best_col])
model = MultinomialNB(alpha=best_alpha)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Accuracy :', round(100 * accuracy_score(y_test, pred), 1))
print('Precision:', round(100 * precision_score(y_test, pred), 1))
print('Recall   :', round(100 * recall_score(y_test, pred), 1))
print('F1       :', round(100 * f1_score(y_test, pred), 1))

Accuracy : 98.9
Precision: 100.0
Recall   : 91.2
F1       : 95.4


## 4. The confusion matrix
The confusion matrix shows the four cases:
- ham that we called ham (correct)
- ham that we called spam (a false alarm: a normal message goes to the spam box)
- spam that we called ham (we missed it: spam stays in the inbox)
- spam that we called spam (correct)

In [4]:
cm = confusion_matrix(y_test, pred)
print(pd.DataFrame(cm, index=['true ham', 'true spam'],
                   columns=['said ham', 'said spam']))
print()
print(classification_report(y_test, pred, target_names=['ham', 'spam'], digits=3))

           said ham  said spam
true ham        901          0
true spam        11        114

              precision    recall  f1-score   support

         ham      0.988     1.000     0.994       901
        spam      1.000     0.912     0.954       125

    accuracy                          0.989      1026
   macro avg      0.994     0.956     0.974      1026
weighted avg      0.989     0.989     0.989      1026



## 5. Save the predictions
Abdullah will compare our two models, so we save what our model said.
We also save the probability, to see how sure the model was.

In [5]:
out = test[['label', 'text']].copy()
out['pred'] = np.where(pred == 1, 'spam', 'ham')
out['prob_spam'] = model.predict_proba(X_test)[:, 1].round(4)
out.to_csv('../data/pred_nb.csv', index=False)
print(out['pred'].value_counts().to_dict())
out.head()

{'ham': 912, 'spam': 114}


,label,text,pred,prob_spam
0,ham,"sir, you will receive the account no another 1...",ham,0.0190
1,ham,You at mu? You should try to figure out how mu...,ham,0.0005
2,ham,The greatest test of courage on earth is to be...,ham,0.0094
3,ham,Gud mrng dear hav a nice day,ham,0.0006
4,ham,I wonder if you'll get this text?,ham,0.0171
